In [1]:
from ultralytics import YOLO
import cv2

## image detection

In [6]:
model = YOLO("yolo26n.pt")
result = model("person.jpg",conf = 0.25,verbose = False)[0]

print("image: ", result.path)
print("Detections: ", len(result.boxes) if result.boxes is not None else 0)


if result.boxes is not None:
    for box, conf, cls in zip(result.boxes.xyxy, result.boxes.conf, result.boxes.cls):
        cid = int(cls)
        print(result.names[cid], 'conf=', round(float(conf),3),"box=", [round(float(v),1) for v in box])

annotated = result.plot()
cv2.imwrite("image_result.jpg", annotated)
cv2.imshow("YOLO image detection", annotated)
cv2.waitKey(0)
cv2.destroyAllWindows()

image:  C:\Users\ajant\Downloads\ProgNxt\Object detection and Tracking Day 2\person.jpg
Detections:  3
horse conf= 0.938 box= [405.4, 138.6, 602.1, 347.9]
person conf= 0.934 box= [191.5, 99.3, 275.3, 374.3]
dog conf= 0.738 box= [64.4, 264.0, 205.4, 345.0]


## object detection in video

In [11]:
from ultralytics import YOLO
import cv2
import time

model = YOLO("yolo11n.pt")
capture = cv2.VideoCapture("slow_traffic_small.mp4")
if not capture.isOpened():
    raise FileNotFoundError(f"Unable to open {INPUT_FILE}")
fps = capture.get(cv2.CAP_PROP_FPS)
width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
output = cv2.VideoWriter( "yolo_detection_workflow_output.mp4", fourcc,fps,(width, height))
frame_count = 0

while True:
    success, frame = capture.read()
    if not success:
        break
    frame_count += 1
    result = model(frame, conf=0.25,imgsz = 640, verbose=False )[0]
    if result.boxes is None:
        for box, conf, cls in zip(result.boxes.xyxy, result.boxes.conf, result.boxes.cls):
            cid = int(cls)
            print(result.names[cid], 'conf=', round(float(conf),3),"box=", [round(float(v),1) for v in box])
    annotated = result.plot()
    output.write(annotated)
    cv2.imshow("YOLO Detection Workflow Application", annotated)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break
capture.release()
output.release()
cv2.destroyAllWindows()
